# 02 NLP Parsing Pipeline
---

## What this notebook does

This notebook implements the **Natural Language Processing (NLP) parsing pipeline**  the entry point of the autonomous task planner system.

The pipeline takes **raw natural language input** from a user and converts it into **structured JSON task objects** that can be fed directly into the RL scheduling engine.

### Examples of what it handles
| Input | Output |
|---|---|
| `"Finish the client report by 3pm, takes about 2 hours"` | `{name, deadline, duration_min, priority, dependencies}` |
| `"Attend the team meeting and send the summary email after"` | Two tasks with dependency link |
| `"Do some work later"` | Task with inferred defaults and vague flag |

### Pipeline architecture
```
User text input
      │
      ▼
 Prompt engineering (system prompt + few-shot examples)
      │
      ▼
 Mistral 7B via Ollama (local, free, offline)
      │
      ▼
 JSON response extraction + validation
      │
      ▼
 Structured task objects (matching dataset schema)
      │
      ▼
 RL Scheduling Engine (next component)
```

### Notebook structure
1. Imports and configuration  
2. Ollama connection check  
3. Prompt engineering  
4. Core parsing function  
5. Single task extraction  
6. Multiple tasks in one sentence  
7. Vague input handling  
8. Dependency detection  
9. Batch evaluation on 100 test cases  
10. Results and accuracy report  


## 1. Imports and Configuration

In [ ]:
import ollama
import json
import re
import uuid
from datetime import datetime, timedelta
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings
warnings.filterwarnings('ignore')

# ── Configuration ─────────────────────────────────────────────────────────────
MODEL          = "mistral"        # Ollama model to use
WORKDAY_START  = "08:00"          # Simulated workday start
WORKDAY_END    = "18:00"          # Simulated workday end
BASE_DATE      = "2025-01-06"     # Simulated Monday (matches dataset)
DEFAULT_DURATION = 60             # Default duration (min) when not specified
DEFAULT_PRIORITY = 3              # Default priority when not specified

# ── Plot style ────────────────────────────────────────────────────────────────
plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor":   "white",
    "axes.spines.top":  False,
    "axes.spines.right":False,
    "axes.grid":        True,
    "grid.alpha":       0.3,
    "font.size":        11,
})
PALETTE = ["#1D9E75", "#534AB7", "#BA7517", "#2E75B6", "#D85A30"]

print("Imports loaded successfully.")
print(f"Model          : {MODEL}")
print(f"Workday        : {WORKDAY_START} – {WORKDAY_END}")
print(f"Base date      : {BASE_DATE}")
print(f"Default dur.   : {DEFAULT_DURATION} min")
print(f"Default priority: {DEFAULT_PRIORITY}")


## 2. Ollama Connection Check

Before running any parsing, we verify that Ollama is running and the Mistral model
is available. If this cell fails, make sure Ollama is running in the background
(check your system tray on Windows).


In [ ]:
def check_ollama_connection():
    """Check if Ollama is running and Mistral is available."""
    try:
        models = ollama.list()
        available = [m['name'] for m in models.get('models', [])]
        print("Ollama is running.")
        print(f"Available models: {available}")
        if any(MODEL in m for m in available):
            print(f"'{MODEL}' model is ready.")
            return True
        else:
            print(f"WARNING: '{MODEL}' not found. Run: ollama pull {MODEL}")
            return False
    except Exception as e:
        print(f"ERROR: Cannot connect to Ollama. Is it running?")
        print(f"Details: {e}")
        return False

connection_ok = check_ollama_connection()


## 3. Prompt Engineering

The quality of the NLP pipeline depends almost entirely on the quality of the prompt.
A well-designed prompt tells the model:
1. Exactly what JSON schema to produce
2. How to handle ambiguous or vague input
3. How to detect dependencies between tasks
4. What defaults to use when information is missing

We use a **system prompt** (defines the model's role and rules) combined with
**few-shot examples** (shows the model exactly what good output looks like).
Few-shot prompting significantly improves structured output consistency
without requiring any fine-tuning.


In [ ]:
SYSTEM_PROMPT = """You are a task extraction assistant for a personal productivity scheduler.
Your job is to extract structured task information from natural language input.

RULES:
1. Extract ALL tasks mentioned in the input — a single sentence may contain multiple tasks.
2. Return ONLY a valid JSON array. No explanations, no markdown, no extra text.
3. Each task must follow this exact schema:
   {
     "id": "T001",                          // sequential, T001 T002 etc.
     "name": "string",                      // short, clear task name (max 8 words)
     "deadline": "2025-01-06THH:MM:SS",     // ISO format, workday is 08:00-18:00
     "deadline_min": integer,               // minutes from 08:00 (0-600)
     "duration_min": integer,               // estimated duration in minutes
     "priority": integer,                   // 1=low, 2=low-med, 3=medium, 4=med-high, 5=high
     "dependencies": [],                    // list of task IDs this task depends on
     "status": "pending",
     "vague": boolean,                      // true if deadline or duration had to be guessed
     "raw_input": "string"                  // the original phrase this task was extracted from
   }

DEADLINE RULES:
- "by 3pm" → 2025-01-06T15:00:00, deadline_min = 420
- "by noon" → 2025-01-06T12:00:00, deadline_min = 240
- "this morning" → 2025-01-06T10:00:00, deadline_min = 120, vague = true
- "later" / "today" / "soon" → 2025-01-06T17:00:00, deadline_min = 540, vague = true
- "end of day" / "EOD" → 2025-01-06T18:00:00, deadline_min = 600

DURATION RULES:
- "takes 2 hours" → 120
- "quick" / "brief" → 15
- "half an hour" → 30
- "about an hour" → 60
- if not mentioned → 60 (default), vague = true

PRIORITY RULES:
- "urgent" / "critical" / "ASAP" → 5
- "important" → 4
- no signal → 3
- "when you have time" / "low priority" → 2
- "if possible" / "optional" → 1

DEPENDENCY RULES:
- "after the meeting, write the summary" → summary depends on meeting
- "once X is done, do Y" → Y depends on X
- "then" / "after" / "once" / "following" → signals dependency

BASE DATE: 2025-01-06 (Monday). Workday: 08:00-18:00.
"""

FEW_SHOT_EXAMPLES = [
    {
        "input": "Finish the client report by 3pm, it should take about 2 hours",
        "output": '[{"id":"T001","name":"Finish client report","deadline":"2025-01-06T15:00:00","deadline_min":420,"duration_min":120,"priority":3,"dependencies":[],"status":"pending","vague":false,"raw_input":"Finish the client report by 3pm, it should take about 2 hours"}]'
    },
    {
        "input": "Attend the team meeting at 10am then write the meeting summary after",
        "output": '[{"id":"T001","name":"Attend team meeting","deadline":"2025-01-06T10:00:00","deadline_min":120,"duration_min":60,"priority":3,"dependencies":[],"status":"pending","vague":false,"raw_input":"Attend the team meeting at 10am"},{"id":"T002","name":"Write meeting summary","deadline":"2025-01-06T17:00:00","deadline_min":540,"duration_min":30,"priority":3,"dependencies":["T001"],"status":"pending","vague":true,"raw_input":"write the meeting summary after"}]'
    },
    {
        "input": "Do some work later",
        "output": '[{"id":"T001","name":"General work task","deadline":"2025-01-06T17:00:00","deadline_min":540,"duration_min":60,"priority":3,"dependencies":[],"status":"pending","vague":true,"raw_input":"Do some work later"}]'
    },
    {
        "input": "Urgently fix the API bug by noon and deploy the update by 2pm",
        "output": '[{"id":"T001","name":"Fix API bug","deadline":"2025-01-06T12:00:00","deadline_min":240,"duration_min":60,"priority":5,"dependencies":[],"status":"pending","vague":false,"raw_input":"Urgently fix the API bug by noon"},{"id":"T002","name":"Deploy update","deadline":"2025-01-06T14:00:00","deadline_min":360,"duration_min":30,"priority":4,"dependencies":[],"status":"pending","vague":false,"raw_input":"deploy the update by 2pm"}]'
    },
]

print("System prompt and few-shot examples defined.")
print(f"System prompt length : {len(SYSTEM_PROMPT)} characters")
print(f"Few-shot examples    : {len(FEW_SHOT_EXAMPLES)}")
print()
print("Few-shot example inputs:")
for i, ex in enumerate(FEW_SHOT_EXAMPLES, 1):
    print(f"  {i}. {ex['input']}")


## 4. Core Parsing Function

The `parse_tasks()` function is the heart of the pipeline. It:
1. Builds the full prompt from the system prompt + few-shot examples + user input
2. Sends it to Mistral via Ollama
3. Extracts and validates the JSON response
4. Returns a list of structured task objects


In [ ]:
def build_prompt(user_input):
    """Build the full prompt with few-shot examples."""
    messages = [{"role": "system", "content": SYSTEM_PROMPT}]
    # Add few-shot examples as conversation turns
    for ex in FEW_SHOT_EXAMPLES:
        messages.append({"role": "user",      "content": ex["input"]})
        messages.append({"role": "assistant", "content": ex["output"]})
    # Add the actual user input
    messages.append({"role": "user", "content": user_input})
    return messages


def extract_json(text):
    """Extract JSON array from model response, handling markdown fences."""
    # Strip markdown code fences if present
    text = re.sub(r'```json\s*', '', text)
    text = re.sub(r'```\s*',     '', text)
    text = text.strip()
    # Find the JSON array
    match = re.search(r'\[.*\]', text, re.DOTALL)
    if match:
        return match.group(0)
    return text


def validate_task(task, index):
    """Validate and fill defaults for a single task object."""
    required = ["id","name","deadline","deadline_min","duration_min","priority","dependencies","status"]
    # Fill missing fields with defaults
    if "id"           not in task: task["id"]           = f"T{index:03d}"
    if "name"         not in task: task["name"]         = "Unnamed task"
    if "deadline"     not in task: task["deadline"]     = f"{BASE_DATE}T17:00:00"
    if "deadline_min" not in task: task["deadline_min"] = 540
    if "duration_min" not in task: task["duration_min"] = DEFAULT_DURATION
    if "priority"     not in task: task["priority"]     = DEFAULT_PRIORITY
    if "dependencies" not in task: task["dependencies"] = []
    if "status"       not in task: task["status"]       = "pending"
    if "vague"        not in task: task["vague"]        = False
    if "raw_input"    not in task: task["raw_input"]    = ""
    # Clamp values to valid ranges
    task["priority"]     = max(1, min(5, int(task["priority"])))
    task["duration_min"] = max(5, min(600, int(task["duration_min"])))
    task["deadline_min"] = max(0, min(600, int(task["deadline_min"])))
    return task


def parse_tasks(user_input, verbose=False):
    """
    Main parsing function.
    Takes natural language input, returns list of structured task dicts.
    """
    if not user_input or not user_input.strip():
        return []

    messages = build_prompt(user_input)

    try:
        response = ollama.chat(model=MODEL, messages=messages)
        raw      = response['message']['content']

        if verbose:
            print("Raw model response:")
            print(raw)
            print()

        json_str = extract_json(raw)
        tasks    = json.loads(json_str)

        if not isinstance(tasks, list):
            tasks = [tasks]

        validated = [validate_task(t, i+1) for i, t in enumerate(tasks)]
        return validated

    except json.JSONDecodeError as e:
        print(f"JSON parse error: {e}")
        print(f"Raw response: {raw}")
        return []
    except Exception as e:
        print(f"Error calling Ollama: {e}")
        return []


def display_tasks(tasks, title="Extracted Tasks"):
    """Pretty print extracted tasks."""
    print(f"\n{'='*60}")
    print(f"  {title} ({len(tasks)} task(s) found)")
    print(f"{'='*60}")
    for t in tasks:
        vague_flag = " [VAGUE]" if t.get("vague") else ""
        dep_str    = f" | depends on: {t['dependencies']}" if t["dependencies"] else ""
        print(f"  ID       : {t['id']}")
        print(f"  Name     : {t['name']}{vague_flag}")
        print(f"  Deadline : {t['deadline']} ({t['deadline_min']} min){dep_str}")
        print(f"  Duration : {t['duration_min']} min")
        print(f"  Priority : {t['priority']}/5")
        print(f"  Raw input: {t.get('raw_input','')}")
        print(f"  {'─'*50}")
    print()


print("Core parsing functions defined.")
print("Ready to parse natural language input.")


## 5. Single Task Extraction

Testing the simplest case — one clear task with a specific deadline and duration.


In [ ]:
# ── Test 1: Single task with clear deadline and duration ──────────────────────
input_1 = "Finish the quarterly report by 3pm, it should take about 2 hours"
print(f"Input: '{input_1}'")
print("Parsing...")

tasks_1 = parse_tasks(input_1)
display_tasks(tasks_1, "Test 1: Single Task")

# Show raw JSON output
print("Raw JSON output:")
print(json.dumps(tasks_1, indent=2))


In [ ]:
# ── Test 2: Single task with urgency signal ───────────────────────────────────
input_2 = "Urgently fix the login bug before noon"
print(f"Input: '{input_2}'")
print("Parsing...")

tasks_2 = parse_tasks(input_2)
display_tasks(tasks_2, "Test 2: Urgent Single Task")


## 6. Multiple Tasks in One Sentence

A single user input may contain several tasks. The pipeline must identify
and extract all of them as separate task objects.


In [ ]:
# ── Test 3: Two tasks in one sentence ────────────────────────────────────────
input_3 = "I need to attend the client meeting at 10am and send the project proposal by 4pm"
print(f"Input: '{input_3}'")
print("Parsing...")

tasks_3 = parse_tasks(input_3)
display_tasks(tasks_3, "Test 3: Two Tasks in One Sentence")


In [ ]:
# ── Test 4: Three tasks ───────────────────────────────────────────────────────
input_4 = "Review the pull request, update the documentation, and deploy the new feature by end of day"
print(f"Input: '{input_4}'")
print("Parsing...")

tasks_4 = parse_tasks(input_4)
display_tasks(tasks_4, "Test 4: Three Tasks in One Sentence")


## 7. Vague Input Handling

Users often provide incomplete information. The pipeline must:
- Extract whatever information is available
- Apply sensible defaults for missing fields
- Flag the task with `vague: true` so the RL agent knows to treat it with lower confidence


In [ ]:
# ── Test 5: Completely vague ─────────────────────────────────────────────────
input_5 = "Do some work later"
print(f"Input: '{input_5}'")
print("Parsing...")

tasks_5 = parse_tasks(input_5)
display_tasks(tasks_5, "Test 5: Vague Input")
print("Notice: vague=True, defaults applied for deadline and duration")


In [ ]:
# ── Test 6: Partial information ───────────────────────────────────────────────
input_6 = "I should probably send that email to the client at some point this morning"
print(f"Input: '{input_6}'")
print("Parsing...")

tasks_6 = parse_tasks(input_6)
display_tasks(tasks_6, "Test 6: Partial Information")


In [ ]:
# ── Test 7: Low priority signal ──────────────────────────────────────────────
input_7 = "If you have time, maybe clean up the old project files"
print(f"Input: '{input_7}'")
print("Parsing...")

tasks_7 = parse_tasks(input_7)
display_tasks(tasks_7, "Test 7: Low Priority Signal")
print(f"Notice: priority={tasks_7[0]['priority']} (low, due to 'if you have time')")


## 8. Dependency Detection

One of the most complex NLP tasks in this pipeline is detecting when one task
must be completed before another can begin. The pipeline looks for linguistic
signals such as: *after*, *then*, *once*, *following*, *when X is done*.


In [ ]:
# ── Test 8: Clear dependency signal ──────────────────────────────────────────
input_8 = "After the team meeting at 10am, write the meeting summary and send it to the team"
print(f"Input: '{input_8}'")
print("Parsing...")

tasks_8 = parse_tasks(input_8)
display_tasks(tasks_8, "Test 8: Dependency Detection")

# Verify dependency was detected
if len(tasks_8) >= 2:
    has_dep = any(len(t["dependencies"]) > 0 for t in tasks_8)
    print(f"Dependency detected: {has_dep}")
    for t in tasks_8:
        if t["dependencies"]:
            print(f"  '{t['name']}' depends on: {t['dependencies']}")


In [ ]:
# ── Test 9: Chain dependency ──────────────────────────────────────────────────
input_9 = "First finish the data analysis, then prepare the presentation based on it, and once that is done present it to the board by 4pm"
print(f"Input: '{input_9}'")
print("Parsing...")

tasks_9 = parse_tasks(input_9)
display_tasks(tasks_9, "Test 9: Chain Dependency")


## 9. Batch Evaluation on 100 Test Cases

To formally evaluate the pipeline's performance, we run it against 100
annotated test cases and measure:

- **Field accuracy**: Are the correct fields extracted?
- **Vague detection rate**: Does it correctly flag ambiguous inputs?
- **Dependency detection rate**: Does it find task dependencies?
- **Multi-task detection rate**: Does it correctly split multi-task inputs?
- **Parse success rate**: Does it return valid JSON every time?

This evaluation provides the numbers for **Section 4.2** of the thesis.


In [ ]:
# ── 100 annotated test cases ─────────────────────────────────────────────────
TEST_CASES = [
    # (input, expected_n_tasks, has_dependency, is_vague, expected_priority)
    # Single clear tasks
    ("Finish the report by 2pm, takes 1 hour",                          1, False, False, 3),
    ("Send the project proposal by end of day",                          1, False, False, 3),
    ("Review the quarterly budget by noon",                              1, False, False, 3),
    ("Update the client presentation by 3pm",                           1, False, False, 3),
    ("Fix the critical database bug immediately",                        1, False, False, 5),
    ("Urgently call the client back before noon",                        1, False, False, 5),
    ("Submit the expense report by 5pm",                                 1, False, False, 3),
    ("Prepare the sprint retrospective by 4pm",                         1, False, False, 3),
    ("Write the technical documentation by end of day",                  1, False, False, 3),
    ("Deploy the hotfix to production by 11am",                         1, False, False, 4),
    # Multi-task inputs
    ("Attend the standup at 9am and review pull requests by noon",       2, False, False, 3),
    ("Send the email and update the spreadsheet by 3pm",                 2, False, False, 3),
    ("Fix the bug and write the test cases by end of day",               2, False, False, 3),
    ("Call the client at 2pm and send the follow-up email after",        2, True,  False, 3),
    ("Prepare the slides and rehearse the presentation by 1pm",          2, False, False, 3),
    ("Review the contract and send feedback by 4pm",                     2, False, False, 3),
    ("Update the API docs and notify the team by noon",                  2, False, False, 3),
    ("Complete the code review and merge the branch by 3pm",             2, False, False, 3),
    ("Backup the database and restart the server by 10am",               2, False, False, 4),
    ("Write the report introduction and conclusion by end of day",       2, False, False, 3),
    # Vague inputs
    ("Do some work later",                                               1, False, True,  3),
    ("I should probably send that email at some point",                  1, False, True,  3),
    ("Finish the project stuff today",                                   1, False, True,  3),
    ("Handle the client thing when you get a chance",                    1, False, True,  2),
    ("Take care of the backlog sometime this morning",                   1, False, True,  3),
    ("Do a quick check on the servers",                                  1, False, True,  3),
    ("Sort out the meeting room booking",                                 1, False, True,  3),
    ("Look into the performance issue when possible",                    1, False, True,  2),
    ("Catch up on emails",                                               1, False, True,  3),
    ("Review some code at some point",                                   1, False, True,  3),
    # Dependency detection
    ("After the morning standup, update the task board",                 2, True,  False, 3),
    ("Once the tests pass, merge the pull request",                      2, True,  False, 3),
    ("Attend the client call then write the action items",               2, True,  False, 3),
    ("Finish the analysis first then prepare the presentation",          2, True,  False, 3),
    ("After fixing the bug, notify the QA team",                        2, True,  False, 3),
    ("Once the report is approved, send it to the client",              2, True,  False, 3),
    ("Complete the research then write the summary by 3pm",             2, True,  False, 3),
    ("Review the PR then deploy if it looks good by noon",              2, True,  False, 3),
    ("Finish the meeting at 11am then send the minutes to the team",    2, True,  False, 3),
    ("After the code review, fix any issues found by end of day",       2, True,  False, 3),
    # Priority signals
    ("If possible, clean up the old log files",                          1, False, False, 1),
    ("When you have time, organise the project folder",                  1, False, False, 2),
    ("Very important: submit the compliance form by 2pm",               1, False, False, 4),
    ("Critical security patch needs to be deployed by 10am",            1, False, False, 5),
    ("Low priority: update the team wiki page",                          1, False, False, 2),
    ("ASAP fix the payment gateway error",                               1, False, False, 5),
    ("Optional: add comments to the legacy code",                        1, False, False, 1),
    ("Urgent: respond to the investor email before noon",               1, False, False, 5),
    ("Not critical but review the old test cases today",                 1, False, True,  2),
    ("Please review the architecture proposal when convenient",          1, False, True,  2),
    # Mixed complexity
    ("Urgently fix the login bug by 10am then notify the team",         2, True,  False, 5),
    ("Review contracts, send proposals, and update CRM by 4pm",        3, False, False, 3),
    ("Attend standup, review PRs, and deploy by noon if tests pass",    3, True,  False, 3),
    ("Do the monthly report sometime today",                             1, False, True,  3),
    ("Critical: patch the server and restart services by 9am",          2, False, False, 5),
    ("After the design review at 2pm, update the mockups",              2, True,  False, 3),
    ("If you have time, prepare backup slides for the presentation",    1, False, True,  1),
    ("Send invoices to all clients by end of day",                       1, False, False, 3),
    ("Once the staging tests pass, promote to production by 3pm",       2, True,  False, 4),
    ("Complete the onboarding doc and share with new team members",     2, False, False, 3),
    # Edge cases
    ("Meeting at 9",                                                     1, False, True,  3),
    ("Email John",                                                       1, False, True,  3),
    ("Fix it",                                                           1, False, True,  3),
    ("Call at 2pm",                                                      1, False, True,  3),
    ("Submit by EOD",                                                    1, False, True,  3),
    ("Presentation at noon",                                             1, False, False, 3),
    ("Daily standup 9am",                                                1, False, False, 3),
    ("Budget review 3pm takes 90 minutes",                              1, False, False, 3),
    ("Review PR #142 by 11am",                                           1, False, False, 3),
    ("Team lunch at 12:30",                                              1, False, False, 3),
    # Professional scenarios
    ("Prepare Q4 financial summary and present to CFO by 2pm",         2, False, False, 4),
    ("Run the nightly database backup before midnight",                  1, False, False, 4),
    ("Review the NDA and send back to legal by end of day",             2, False, False, 3),
    ("Coordinate with the offshore team at 8am then summarise actions", 2, True,  False, 3),
    ("Finish the sprint planning and update Jira by noon",              2, False, False, 3),
    ("Debug the memory leak and push a fix by 3pm",                     2, False, False, 4),
    ("Write release notes and tag the version by end of day",           2, False, False, 3),
    ("After the client call, update the CRM and send a follow-up",     3, True,  False, 3),
    ("Do a security audit of the new endpoints before deployment",      2, True,  False, 4),
    ("Review the analytics dashboard and share insights by 4pm",       2, False, False, 3),
    # More vague/low-priority
    ("Tidy up the shared drive at some point",                          1, False, True,  2),
    ("Look at the competitor analysis when you get a chance",           1, False, True,  2),
    ("Maybe reorganise the team wiki",                                  1, False, True,  2),
    ("Think about the new feature ideas",                               1, False, True,  2),
    ("Catch up on Slack messages",                                      1, False, True,  3),
    ("Glance at the server logs",                                       1, False, True,  3),
    ("Something about the deployment",                                  1, False, True,  3),
    ("The report thing",                                                1, False, True,  3),
    ("Handle the ticket",                                               1, False, True,  3),
    ("Check on the new hire's progress",                                1, False, True,  3),
    # More dependencies
    ("Once the build succeeds, run the integration tests",              2, True,  False, 4),
    ("After reviewing the budget, get CFO approval by 3pm",             2, True,  False, 4),
    ("Finish the wireframes then hand off to the dev team",             2, True,  False, 3),
    ("After QA signs off, schedule the release for 4pm",               2, True,  False, 4),
    ("Review the proposal then schedule a follow-up meeting",           2, True,  False, 3),
    ("Complete the migration then run validation checks by noon",       2, True,  False, 4),
    ("After the morning briefing, assign tasks to the team",            2, True,  False, 3),
    ("Once the invoice is approved, process the payment by EOD",        2, True,  False, 3),
    ("Finish the prototype then demo it to stakeholders at 3pm",        2, True,  False, 3),
    ("After the code freeze, prepare the release checklist",            2, True,  False, 4),
]

print(f"Total test cases: {len(TEST_CASES)}")
print("Starting batch evaluation... (this may take 3-5 minutes)")
print("Each dot = 1 test case completed")


In [ ]:
# ── Run batch evaluation ─────────────────────────────────────────────────────
results = []

for i, (inp, exp_n, exp_dep, exp_vague, exp_priority) in enumerate(TEST_CASES):
    try:
        tasks = parse_tasks(inp, verbose=False)

        # Metrics
        parse_success  = len(tasks) > 0
        n_tasks        = len(tasks)
        task_count_ok  = n_tasks == exp_n
        has_dep        = any(len(t["dependencies"]) > 0 for t in tasks)
        dep_ok         = has_dep == exp_dep
        is_vague       = any(t.get("vague", False) for t in tasks)
        vague_ok       = is_vague == exp_vague
        priority_ok    = tasks[0]["priority"] == exp_priority if tasks else False
        all_fields_ok  = all(
            all(f in t for f in ["id","name","deadline","deadline_min",
                                  "duration_min","priority","dependencies","status"])
            for t in tasks
        ) if tasks else False

        results.append({
            "input":          inp,
            "parse_success":  parse_success,
            "expected_tasks": exp_n,
            "got_tasks":      n_tasks,
            "task_count_ok":  task_count_ok,
            "dep_expected":   exp_dep,
            "dep_detected":   has_dep,
            "dep_ok":         dep_ok,
            "vague_expected": exp_vague,
            "vague_detected": is_vague,
            "vague_ok":       vague_ok,
            "priority_ok":    priority_ok,
            "all_fields_ok":  all_fields_ok,
        })
        print(".", end="", flush=True)

    except Exception as e:
        results.append({
            "input": inp, "parse_success": False,
            "expected_tasks": exp_n, "got_tasks": 0,
            "task_count_ok": False, "dep_ok": False,
            "vague_ok": False, "priority_ok": False, "all_fields_ok": False,
        })
        print("X", end="", flush=True)

print(f"\n\nEvaluation complete. {len(results)} test cases processed.")
df_results = pd.DataFrame(results)


## 10. Evaluation Results and Accuracy Report

This section presents the pipeline's performance across all 100 test cases.
These figures will be reported in **Section 4.2** of the thesis.


In [ ]:
# ── Summary statistics ───────────────────────────────────────────────────────
metrics = {
    "Parse success rate":       df_results["parse_success"].mean()  * 100,
    "Field completeness":       df_results["all_fields_ok"].mean()  * 100,
    "Task count accuracy":      df_results["task_count_ok"].mean()  * 100,
    "Dependency detection":     df_results["dep_ok"].mean()         * 100,
    "Vague detection":          df_results["vague_ok"].mean()       * 100,
    "Priority accuracy":        df_results["priority_ok"].mean()    * 100,
}

print("=" * 50)
print("  NLP PIPELINE EVALUATION RESULTS")
print("  100 annotated test cases")
print("=" * 50)
for metric, value in metrics.items():
    bar = "█" * int(value / 5)
    print(f"  {metric:<25} {value:>5.1f}%  {bar}")
print("=" * 50)

overall = sum(metrics.values()) / len(metrics)
print(f"  Overall accuracy         {overall:>5.1f}%")
print("=" * 50)


In [ ]:
# ── Visualisation ────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart of all metrics
metric_names  = list(metrics.keys())
metric_values = list(metrics.values())
bars = axes[0].barh(metric_names, metric_values, color=PALETTE[1],
                    edgecolor="white", height=0.6)
axes[0].set_xlim(0, 110)
axes[0].set_xlabel("Accuracy (%)")
axes[0].set_title("NLP Pipeline — Metric Breakdown", fontweight="bold")
axes[0].axvline(80, color=PALETTE[2], linestyle="--", linewidth=1.5,
                label="80% threshold")
axes[0].legend(fontsize=9)
for bar, val in zip(bars, metric_values):
    axes[0].text(val + 1, bar.get_y() + bar.get_height()/2,
                 f"{val:.1f}%", va="center", fontsize=10)

# Pie chart: pass vs fail on task count accuracy
pass_count = df_results["task_count_ok"].sum()
fail_count = len(df_results) - pass_count
axes[1].pie([pass_count, fail_count],
            labels=[f"Correct count
({pass_count})", f"Wrong count
({fail_count})"],
            colors=[PALETTE[0], PALETTE[4]], autopct="%1.1f%%",
            startangle=90, wedgeprops={"edgecolor":"white","linewidth":2})
axes[1].set_title("Task Count Accuracy
(100 test cases)", fontweight="bold")

plt.tight_layout()
plt.savefig("data/fig6_nlp_evaluation.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figure saved: data/fig6_nlp_evaluation.png")


In [ ]:
# ── Error analysis ───────────────────────────────────────────────────────────
print("\nFailed cases analysis:")
print("-" * 60)

failed = df_results[~df_results["task_count_ok"]]
if len(failed) > 0:
    print(f"Task count errors ({len(failed)} cases):")
    for _, row in failed.head(5).iterrows():
        print(f"  Expected {row['expected_tasks']} tasks, got {row['got_tasks']}")
        print(f"  Input: '{row['input'][:70]}...'")
        print()
else:
    print("No task count errors!")

print(f"\nDependency detection errors: {(~df_results['dep_ok']).sum()}")
print(f"Vague detection errors     : {(~df_results['vague_ok']).sum()}")
print(f"Priority errors            : {(~df_results['priority_ok']).sum()}")


In [ ]:
# ── Final summary ────────────────────────────────────────────────────────────
print("=" * 55)
print("  NLP PIPELINE — COMPLETE")
print("=" * 55)
print(f"  Model used      : Mistral 7B (via Ollama, local)")
print(f"  Test cases      : {len(TEST_CASES)}")
print(f"  Parse success   : {df_results['parse_success'].mean()*100:.1f}%")
print(f"  Task accuracy   : {df_results['task_count_ok'].mean()*100:.1f}%")
print(f"  Dep. detection  : {df_results['dep_ok'].mean()*100:.1f}%")
print(f"  Vague detection : {df_results['vague_ok'].mean()*100:.1f}%")
print()
print("  Output schema matches dataset schema: YES")
print("  Ready to feed into RL environment  : YES")
print()
print("  Next step → 03_Baselines.ipynb")
print("=" * 55)
